Run: docker compose up -d

In [25]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointsSelector, PointIdsList

client = QdrantClient(host="qdrant_nosql_lab", port=6333)


### Tworzenie / nadpisanie kolekcji

In [26]:
client.recreate_collection(
    collection_name="my_collection",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)


/tmp/ipykernel_6443/332858613.py:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

### Sprawdzanie czy kolekcja istnieje

In [27]:
exists = client.collection_exists("my_collection")
print("Istnieje:", exists)


Istnieje: True


### Lista wszystkich kolekcji

In [28]:
collections = client.get_collections()
print(collections)

collections=[CollectionDescription(name='my_collection')]


### Dodawanie punktów (danych)

In [29]:
from qdrant_client.models import PointStruct

# Przygotowanie punktów do dodania:
# Każdy punkt to obiekt PointStruct z:
# - unikalnym ID (np. 0, 1, 2...),
# - wektorem (embeddingiem, np. 384-wymiarowym),
# - opcjonalnym payloadem (np. tekst lub metadane)

points = [
    PointStruct(
        id=0,                                # unikalny identyfikator punktu
        vector=[0.1] * 384,                  # 384-wymiarowy wektor (np. embedding zdania)
        payload={"text": "przykład 1"}       # dodatkowe dane (można filtrować po payloadach)
    ),
    PointStruct(
        id=1,
        vector=[0.2] * 384,
        payload={"text": "przykład 2"}
    ),
    PointStruct(
        id=2,
        vector=[0.3] * 384,
        payload={"text": "przykład 3"}
    ),
    PointStruct(
        id=3,
        vector=[0.4] * 384,
        payload={"text": "przykład 4"}
    ),
    PointStruct(
        id=4,
        vector=[0.5] * 384,
        payload={"text": "przykład 5"}
    ),
    PointStruct(
        id=5,
        vector=[0.6] * 384,
        payload={"text": "przykład 6"}
    ),
    PointStruct(
        id=6,
        vector=[0.7] * 384,
        payload={"text": "przykład 7"}
    ),
    PointStruct(
        id=7,
        vector=[0.8] * 384,
        payload={"text": "przykład 8"}
    ),
    PointStruct(
        id=8,
        vector=[0.9] * 384,
        payload={"text": "przykład 9"}
    ),
    PointStruct(
        id=9,
        vector=[0.1] * 384,
        payload={"text": "przykład 10"}
    )
]

# Dodanie punktów do kolekcji o nazwie 'my_collection'
# Jeśli punkty o tych ID już istnieją – zostaną nadpisane (upsert = insert + update)
client.upsert(
    collection_name="my_collection",
    points=points
)


UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

### Wyszukiwanie podobnych wektorów

In [30]:
query_vector = [0.1]*384 # Wektor o długości 384 i wartościach 0.1

results = client.query_points(
    collection_name="my_collection",
    query=query_vector,
    limit=3,
    with_payload=True
)

for result in results:
    # print(result)
    for item in result[1]:
        # print(item)
        text = item.payload["text"]
        score = item.score
        # print(f"- \"{text}\" (similarity: {score:.4f})")
        print(f"ID: {item.id}, score: {item.score}, payload: {item.payload}")


ID: 6, score: 1.0000002, payload: {'text': 'przykład 7'}
ID: 9, score: 0.9999999, payload: {'text': 'przykład 10'}
ID: 4, score: 0.9999999, payload: {'text': 'przykład 5'}


### Usuwanie punktów

In [31]:
client.delete(
    collection_name="my_collection",
    points_selector=PointIdsList(points=[0, 1])
)


UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

### Usuwanie kolekcji

In [32]:
client.delete_collection(collection_name="my_collection")

True